In [ ]:
"""総合演習2 社会系シミュレーション 第3回 (2026-11-25)

SIRモデルの可動エージェントシミュレーション
"""

import random
from typing import Final, Self

import matplotlib.pyplot as plt
from IPython.display import HTML, display
from matplotlib import animation
from matplotlib.artist import Artist


# ========== パラメータ ==========
NUM_AGENTS: Final[int] = 100      # エージェントの数
X_MIN: Final[float] = 0           # x座標の最小値
X_MAX: Final[float] = 10          # x座標の最大値
Y_MIN: Final[float] = 0           # y座標の最小値
Y_MAX: Final[float] = 10          # y座標の最大値
DX_ABS: Final[float] = 0.05       # x方向の移動距離の上限
DY_ABS: Final[float] = 0.05       # y方向の移動距離の上限
RECOVERY_STEP: Final[int] = 1000  # 感染から回復までのステップ数

NUM_STEPS: Final[int] = 2000      # 時間ステップ数
PLOT_INTERVAL: Final[int] = 50    # 可視化するステップ間隔
# ==============================


LX: Final[float] = X_MAX - X_MIN
LY: Final[float] = Y_MAX - Y_MIN
MARGIN: Final[float] = 0.1

STATE_COLOR: Final[dict[str, str]] = {
    'susceptible': 'blue',
    'infected': 'red',
    'recovered': 'green'
}
STATE_MARKER: Final[dict[str, str]] = {
    'susceptible': 'o',
    'infected': '^',
    'recovered': 's'
}


class MovableAgent:
    """可動エージェントを表すクラス

    Attributes
    ----------
    x : float
        エージェントのx座標
    y : float
        エージェントのy座標
    status : str
        エージェントの感染状態 ('susceptible', 'infected', 'recovered')
    infected_step : int
        感染してからのステップ数
    """

    def __init__(self, init_x: float, init_y: float, init_status: str) -> None:
        """MovableAgentクラスのインスタンスのイニシャライザ

        Parameters
        ----------
        init_x : float
            エージェントの初期x座標
        init_y : float
            エージェントの初期y座標
        init_status : str
            エージェントの初期感染状態 ('susceptible', 'infected', 'recovered')
        """

        self.x: float = init_x
        self.y: float = init_y
        self.status: str = init_status
        self.infected_step: int = 0

    def move(self) -> None:
        """エージェントを1ステップ移動させる

        Notes
        -----
        エージェントの移動はランダムに決定
        """

        dx: float = DX_ABS * 2*(random.random() - 0.5)
        dy: float = DY_ABS * 2*(random.random() - 0.5)

        self.x += dx
        self.y += dy

        # 範囲外に出たら範囲内に戻す
        if self.x < X_MIN:
            self.x = X_MIN
        elif self.x > X_MAX:
            self.x = X_MAX
        if self.y < Y_MIN:
            self.y = Y_MIN
        elif self.y > Y_MAX:
            self.y = Y_MAX

    def change_status(self) -> None:
        """エージェントの感染状態を更新する"""

        if self.status == 'susceptible':
            pass

        if self.status == 'infected':
            self.infected_step += 1
            if self.infected_step >= RECOVERY_STEP:
                self.status = 'recovered'

    def calc_distance(self, other: Self) -> float:
        """他のエージェントとの距離を計算する

        Parameters
        ----------
        other : MovableAgent
            距離を計算する相手のエージェント

        Returns
        -------
        distance : float
            他のエージェントとの距離
        """

        difference_x: float = self.x - other.x
        difference_y: float = self.y - other.y
        distance: float = (difference_x**2 + difference_y**2)**0.5

        return distance


class MovableAgentSystem:
    """可動エージェント全体を表すクラス

    Attributes
    ----------
    num_agents : int
        エージェントの数
    agents : list[MovableAgent]
        全ての可動エージェントを格納するリスト
    timestep : float
        現在の時間ステップ
    """

    def __init__(self, num_agents: int) -> None:
        """MovableAgentSystemクラスのインスタンスのイニシャライザ

        Parameters
        ----------
        num_agents : int
            エージェントの数

        Notes
        -----
        エージェントの初期位置はランダムに設定
        """

        self.num_agents: int = num_agents
        self.agents: list[MovableAgent] = []
        self.timestep: int = 0

        for i in range(num_agents):
            init_x: float = X_MIN + (X_MAX - X_MIN) * random.random()
            init_y: float = Y_MIN + (Y_MAX - Y_MIN) * random.random()

            init_status: str = 'susceptible'
            if i == 0:
                init_status = 'infected'

            self.agents.append(MovableAgent(init_x, init_y, init_status))

    def time_evolution(self) -> None:
        """システムを1ステップ時間発展させる"""

        for agent in self.agents:
            agent.change_status()
            agent.move()

        self.timestep += 1

    def visualize(self) -> list[Artist]:
        """各タイムステップでのエージェントの位置を可視化する

        Returns
        -------
        frame : list[Artist]
            可視化結果のArtistオブジェクトのリスト

        Notes
        -----
        matplotlib.animation.ArtistAnimationを使用
        """

        frame: list[Artist] = [
            ax.text(X_MIN, Y_MAX+1.5*MARGIN*LY, f'time step = {self.timestep}')
        ]
        for agent in self.agents:
            frame += ax.plot(agent.x, agent.y,
                             color=STATE_COLOR[agent.status],
                             marker=STATE_MARKER[agent.status])

        return frame


if __name__ == '__main__':

    # 可動エージェントを初期化
    system: MovableAgentSystem = MovableAgentSystem(NUM_AGENTS)

    # 可視化の準備
    fig, ax = plt.subplots(figsize=(5, 5), dpi=150)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_xlim(X_MIN-MARGIN*LX, X_MAX+MARGIN*LX)
    ax.set_ylim(Y_MIN-MARGIN*LY, Y_MAX+MARGIN*LY)
    ax.grid()
    ax.set_aspect('equal')
    for status in STATE_COLOR.keys():
        ax.plot([], [], 
                color=STATE_COLOR[status], marker=STATE_MARKER[status], linestyle='None', label=status)
    ax.legend(loc='upper right', bbox_to_anchor=(1.1, 1.15))

    # 壁の描画
    ax.axvspan(X_MIN-MARGIN*LX, X_MIN, color='gray')
    ax.axvspan(X_MAX, X_MAX+MARGIN*LX, color='gray')
    ax.axhspan(Y_MIN-MARGIN*LY, Y_MIN, color='gray')
    ax.axhspan(Y_MAX, Y_MAX+MARGIN*LY, color='gray')

    # 初期時刻の可視化
    artists: list[list[Artist]] = []
    artists.append(system.visualize())

    # 時間発展
    for _ in range(NUM_STEPS):
        system.time_evolution()

        if system.timestep % PLOT_INTERVAL == 0:
            artists.append(system.visualize())

    # アニメーションの作成
    anim = animation.ArtistAnimation(fig, artists)
    plt.close(fig)
    display(HTML(anim.to_jshtml()))
    anim.save('./social_3rd_sir.gif', writer="pillow")